# ĐỒ ÁN MÔN HỌC MÁY CHO BẢO MẬT
## Đề tài: Ứng dụng Học máy trong tự động nhận diện và phân loại liên kết Phishing

**Thuật toán sử dụng:**
1. **Random Forest Classifier**
2. **XGBoost (Extreme Gradient Boosting)**
3. **Support Vector Machine (SVM)**

---

In [ ]:
import os
import sys
import re
import time
import joblib
import urllib.parse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
import tldextract

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc
import xgboost as xgb

# Thiết lập đồ họa
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print('[OK] Da import tat ca cac thu vien thanh cong!')

## 1. Định nghĩa Hàm Tiền xử lý & Trích xuất 15 Đặc trưng URL

In [ ]:
def unshorten_single(url, timeout=2.5):
    if not isinstance(url, str) or not url.strip():
        return url, False
    url_str = url.strip()
    url_lower = url_str.lower()
    url_for_parse = url_str if (url_lower.startswith('http://') or url_lower.startswith('https://')) else 'http://' + url_str
    try:
        parsed = urllib.parse.urlparse(url_for_parse)
        path = parsed.path
        ext = tldextract.extract(url_for_parse)
        registered_domain = getattr(ext, 'top_domain_under_public_suffix', getattr(ext, 'registered_domain', ''))
        domain_name = ext.domain
        suffix = ext.suffix
        full_domain = f"{domain_name}.{suffix}".lower() if suffix else domain_name.lower()
    except Exception:
        path = ""
        registered_domain = ""
        full_domain = ""
    is_shortener_domain = (full_domain in SHORTENING_SERVICES or registered_domain.lower() in SHORTENING_SERVICES)
    looks_like_shortener_path = (len(path.strip('/')) > 0 and len(path.strip('/')) <= 20 and '.' not in path)
    if not (is_shortener_domain or looks_like_shortener_path):
        return url_str, False
    curr_url = url_str
    expanded = False
    try:
        url_req = curr_url if (curr_url.lower().startswith('http://') or curr_url.lower().startswith('https://')) else 'http://' + curr_url
        import requests
        resp = requests.head(url_req, allow_redirects=True, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
        if resp.url and resp.url.strip().rstrip('/') != curr_url.strip().rstrip('/'):
            curr_url = resp.url.strip()
            expanded = True
    except Exception:
        try:
            url_req = curr_url if (curr_url.lower().startswith('http://') or curr_url.lower().startswith('https://')) else 'http://' + curr_url
            import requests
            resp = requests.get(url_req, allow_redirects=True, stream=True, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
            if resp.url and resp.url.strip().rstrip('/') != curr_url.strip().rstrip('/'):
                curr_url = resp.url.strip()
                expanded = True
        except Exception:
            pass
    if not expanded:
        try:
            url_req = curr_url if (curr_url.lower().startswith('http://') or curr_url.lower().startswith('https://')) else 'http://' + curr_url
            import urllib.request
            req = urllib.request.Request(url_req, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                next_u = r.geturl()
                if next_u and next_u.strip().rstrip('/') != curr_url.strip().rstrip('/'):
                    curr_url = next_u.strip()
                    expanded = True
        except Exception:
            pass
    return curr_url, expanded

def unshorten_url(url, timeout=2.5, max_chain=3):
    curr = url
    any_expanded = False
    for _ in range(max_chain):
        next_u, exp = unshorten_single(curr, timeout=timeout)
        if exp:
            curr = next_u
            any_expanded = True
        else:
            break
    return curr, any_expanded

def is_ip_address(hostname):
    if not hostname:
        return 0
    ipv4_pattern = r'^(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$'
    if re.match(ipv4_pattern, hostname):
        return 1
    if re.match(r'^0x[0-9a-fA-F]+', hostname) or re.match(r'^[0-9]+$', hostname):
        return 1
    return 0

def extract_features(url, auto_unshorten=True):
    if not url or not isinstance(url, str):
        url = ""
    original_url = url.strip()
    target_url = original_url
    unshortened_url = None
    was_shortened = 0
    if auto_unshorten and original_url:
        expanded_url, is_exp = unshorten_url(original_url, timeout=2.5)
        if is_exp:
            target_url = expanded_url
            unshortened_url = expanded_url
            was_shortened = 1
        else:
            url_for_parse_init = original_url if (original_url.lower().startswith('http://') or original_url.lower().startswith('https://')) else 'http://' + original_url
            try:
                ext_i = tldextract.extract(url_for_parse_init)
                reg_dom_i = getattr(ext_i, 'top_domain_under_public_suffix', getattr(ext_i, 'registered_domain', ''))
                full_dom_i = f"{ext_i.domain}.{ext_i.suffix}".lower() if ext_i.suffix else ext_i.domain.lower()
                if full_dom_i in SHORTENING_SERVICES or reg_dom_i.lower() in SHORTENING_SERVICES:
                    was_shortened = 1
            except Exception:
                pass
    url_str = target_url
    url_lower = url_str.lower()
    url_for_parse = url_str if (url_lower.startswith('http://') or url_lower.startswith('https://')) else 'http://' + url_str
    parsed = None
    try:
        parsed = urllib.parse.urlparse(url_for_parse)
        hostname = parsed.netloc or parsed.path.split('/')[0]
    except Exception:
        hostname = url_str.split('/')[0]
    try:
        ext = tldextract.extract(url_for_parse)
        subdomain = ext.subdomain
        registered_domain = getattr(ext, 'top_domain_under_public_suffix', getattr(ext, 'registered_domain', ''))
        domain_name = ext.domain
        suffix = ext.suffix
    except Exception:
        subdomain = ""
        registered_domain = ""
        domain_name = ""
        suffix = ""
    url_unquoted = urllib.parse.unquote(url_str)
    url_length = len(url_unquoted)
    subdomain_count = len([p for p in subdomain.split('.') if p]) if subdomain else 0
    subdomain_abuse = 1 if subdomain_count > 3 else 0
    has_ip = is_ip_address(hostname.split(':')[0])
    last_double_slash = url_lower.rfind('//')
    abnormal_double_slash = 1 if last_double_slash > 7 else 0
    has_at_symbol = 1 if '@' in url_str else 0
    full_domain = f"{domain_name}.{suffix}".lower() if suffix else domain_name.lower()
    is_shortened = 1 if (was_shortened or full_domain in SHORTENING_SERVICES or registered_domain.lower() in SHORTENING_SERVICES) else 0
    security_keyword_count = sum(1 for kw in SECURITY_KEYWORDS if kw in url_lower)
    has_security_keywords = 1 if security_keyword_count > 0 else 0
    OFFICIAL_TRUSTED_SUFFIXES = {'gov.vn', 'gov', 'edu.vn', 'edu', 'chinhphu.vn'}
    is_official_domain = 1 if (registered_domain.lower() in OFFICIAL_LEGITIMATE_DOMAINS or suffix.lower() in OFFICIAL_TRUSTED_SUFFIXES) else 0
    targets_brand = 0
    if not is_official_domain:
        for brand in TARGETED_BRANDS:
            if brand in url_lower:
                if domain_name.lower() != brand:
                    targets_brand = 1
                    break
    host_path = (parsed.netloc + parsed.path) if (parsed is not None and hasattr(parsed, 'netloc') and parsed.netloc) else url_unquoted.split('?')[0]
    host_path_length = len(host_path)
    count_dots = url_str.count('.')
    count_hyphens = url_str.count('-')
    count_digits = sum(c.isdigit() for c in url_unquoted)
    res = {
        'url_length': url_length,
        'host_path_length': host_path_length,
        'subdomain_count': subdomain_count,
        'subdomain_abuse': subdomain_abuse,
        'has_ip': has_ip,
        'abnormal_double_slash': abnormal_double_slash,
        'has_at_symbol': has_at_symbol,
        'is_shortened': is_shortened,
        'has_security_keywords': has_security_keywords,
        'security_keyword_count': security_keyword_count,
        'targets_brand': targets_brand,
        'is_official_domain': is_official_domain,
        'count_dots': count_dots,
        'count_hyphens': count_hyphens,
        'count_digits': count_digits
    }
    if unshortened_url:
        res['unshortened_url'] = unshortened_url
    return res

print('[OK] Da dinh nghia ham unshorten_single, unshorten_url va extract_features thanh cong!')


## 2. Tải & Kiểm tra Bộ Dữ liệu Đặc trưng

In [ ]:
data_path = os.path.abspath(os.path.join("..", "data", "processed", "features_extracted.csv"))
os.makedirs(os.path.dirname(data_path), exist_ok=True)

if not os.path.exists(data_path):
    print("[!] Tep features_extracted.csv chua ton tai. Dang tim tep du lieu tho de tu dong trich xuat dac trung...")
    raw_paths = [
        os.path.abspath(os.path.join("..", "data", "raw", "phishing_site_urls.csv.zip")),
        os.path.abspath(os.path.join("..", "data", "raw", "phishing_site_urls.zip")),
        os.path.abspath(os.path.join("..", "data", "raw", "phishing_site_urls.csv")),
        os.path.abspath(os.path.join("..", "phishing_site_urls.csv.zip")),
        os.path.abspath(os.path.join("..", "phishing_site_urls.csv")),
        os.path.abspath(os.path.join("..", "data", "phishing_site_urls.csv"))
    ]
    raw_data_path = None
    for p in raw_paths:
        if os.path.exists(p):
            raw_data_path = p
            break
            
    if raw_data_path:
        print(f"[*] Tim thay tep du lieu tho tai: {raw_data_path}")
        print("[*] Dang doc tep va trich xuat 15 dac trung cho toan bo URLs (Qua trinh nay co the mat vai phut)...")
        raw_df = pd.read_csv(raw_data_path)
        url_col = "URL" if "URL" in raw_df.columns else raw_df.columns[0]
        label_col = "Label" if "Label" in raw_df.columns else raw_df.columns[1]
        raw_df["Label_encoded"] = raw_df[label_col].map(lambda x: 1 if str(x).lower() in ["bad", "phishing", "1"] else 0)
        
        print("[*] Dang trich xuat dac trung...")
        features_list = [extract_features(u) for u in raw_df[url_col]]
        features_df = pd.DataFrame(features_list)
        df = pd.concat([raw_df[[url_col, label_col, "Label_encoded"]], features_df], axis=1)
        df.to_csv(data_path, index=False)
        print(f"[✓] Da trich xuat va luu xong tep dac trung vao: {data_path}")
    else:
        raise FileNotFoundError("[X] Khong tim thay tep du lieu tho (phishing_site_urls.csv.zip) de tu dong trich xuat dac trung!")
else:
    print(f"[*] Dang tai du lieu dac trung san co tu: {data_path}")
    df = pd.read_csv(data_path)

print(f"[*] Kich thuoc bo du lieu: {df.shape}")
df.head(10)


### 2.1 Trực quan hóa Phân bố Nhãn Dữ liệu

In [ ]:
figures_dir = os.path.abspath(os.path.join("..", "reports", "figures"))
os.makedirs(figures_dir, exist_ok=True)

plt.figure(figsize=(7, 5))
counts = df['Label_encoded'].value_counts()
labels = ['Benign (0)', 'Phishing (1)']
colors = ['#22c55e', '#ef4444']
bars = plt.bar(labels, [counts.get(0, 0), counts.get(1, 0)], color=colors, width=0.5, edgecolor='black', alpha=0.85)
for bar in bars:
    height = bar.get_height()
    percentage = (height / len(df)) * 100
    plt.text(bar.get_x() + bar.get_width()/2., height + (max(counts)*0.01), f'{height:,}\n({percentage:.1f}%)', ha='center', va='bottom', fontweight='bold')
plt.title('Phan bo Nhan trong Bo Du lieu Phishing URL', fontweight='bold')
plt.ylabel('So luong URLs')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'label_distribution.png'), dpi=300)
plt.show()

### 2.2 Ma trận Tương quan Đặc trưng (Correlation Heatmap)

In [ ]:
plt.figure(figsize=(12, 9))
cols = FEATURE_COLUMNS + ['Label_encoded']
corr = df[cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(230, 20, as_cmap=True), vmax=1.0, vmin=-1.0, center=0, square=True, linewidths=.5, annot=True, fmt=".2f", cbar_kws={"shrink": .8})
plt.title('Biieu do Ma tran Tuong quan Dac trung (Heatmap)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'feature_correlation_heatmap.png'), dpi=300)
plt.show()

## 3. Huấn luyện & Đánh giá 3 Mô hình Học máy

In [ ]:
X = df[FEATURE_COLUMNS]
y = df['Label_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models_dir = os.path.abspath(os.path.join("..", "models"))
os.makedirs(models_dir, exist_ok=True)
joblib.dump(scaler, os.path.join(models_dir, "scaler.joblib"))

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_split=5, n_jobs=-1, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, max_depth=10, learning_rate=0.1, n_jobs=-1, random_state=42, eval_metric='logloss'),
    "Support Vector Machine (SVM)": CalibratedClassifierCV(SGDClassifier(loss='hinge', penalty='l2', max_iter=1000, random_state=42, n_jobs=-1))
}

results = []
models_eval = {}

for name, model in models.items():
    t0 = time.time()
    if "SVM" in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    t_elapsed = time.time() - t0
    
    models_eval[name] = {'y_pred': y_pred, 'y_prob': y_prob}
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_val = roc_auc_score(y_test, y_prob)
    
    save_filename = name.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".joblib"
    joblib.dump(model, os.path.join(models_dir, save_filename))
    
    results.append({
        "Mo hinh": name,
        "Accuracy": f"{acc*100:.2f}%",
        "Precision": f"{prec*100:.2f}%",
        "Recall": f"{rec*100:.2f}%",
        "F1-Score": f"{f1*100:.2f}%",
        "ROC-AUC": f"{auc_val*100:.2f}%",
        "Thoi gian (s)": f"{t_elapsed:.2f}"
    })

results_df = pd.DataFrame(results)
results_df

### 3.1 Ma trận Nhầm lẫn (Confusion Matrices)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['Blues', 'Oranges', 'Greens']
for i, (name, data) in enumerate(models_eval.items()):
    cm = confusion_matrix(y_test, data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=colors[i], ax=axes[i], cbar=False, xticklabels=['Benign (0)', 'Phishing (1)'], yticklabels=['Benign (0)', 'Phishing (1)'], annot_kws={"size": 14, "weight": "bold"})
    axes[i].set_title(f'Confusion Matrix: {name}', fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'confusion_matrices.png'), dpi=300)
plt.show()

### 3.2 Đường cong ROC-AUC

In [ ]:
plt.figure(figsize=(8, 6))
colors_dict = {'Random Forest': '#2563eb', 'XGBoost': '#dc2626', 'Support Vector Machine (SVM)': '#16a34a'}
for name, data in models_eval.items():
    fpr, tpr, _ = roc_curve(y_test, data['y_prob'])
    roc_auc_v = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors_dict.get(name, 'blue'), lw=2, label=f'{name} (AUC = {roc_auc_v:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
plt.title('So sanh Duong cong ROC (ROC-AUC Curves)', fontweight='bold')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'roc_curves_comparison.png'), dpi=300)
plt.show()

### 3.3 Biểu đồ Tầm quan trọng Đặc trưng (Feature Importance)

In [ ]:
rf_m = models["Random Forest"]
xgb_m = models["XGBoost"]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
pd.Series(rf_m.feature_importances_, index=FEATURE_COLUMNS).sort_values().plot(kind='barh', ax=axes[0], color='#2563eb')
axes[0].set_title('Feature Importance - Random Forest', fontweight='bold')
pd.Series(xgb_m.feature_importances_, index=FEATURE_COLUMNS).sort_values().plot(kind='barh', ax=axes[1], color='#dc2626')
axes[1].set_title('Feature Importance - XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'feature_importance.png'), dpi=300)
plt.show()

## 4. Kiểm thử Dự đoán URL Trực tuyến (Single & Batch Inference)

### 4.1 Kiểm thử URL Đơn lẻ

In [ ]:
def predict_url_phishing(url):
    rf = models["Random Forest"]
    xgb_m = models["XGBoost"]
    svm_m = models["Support Vector Machine (SVM)"]
    
    feats = extract_features(url)
    input_df = pd.DataFrame([feats])[FEATURE_COLUMNS]
    input_scaled = scaler.transform(input_df)
    
    prob_rf = rf.predict_proba(input_df)[0][1]
    prob_xgb = xgb_m.predict_proba(input_df)[0][1]
    prob_svm = svm_m.predict_proba(input_scaled)[0][1]
    
    avg_prob = (prob_rf + prob_xgb + prob_svm) / 3
    is_phishing = avg_prob >= 0.5
    
    if feats['is_official_domain'] == 1 and feats['has_ip'] == 0 and feats['is_shortened'] == 0 and feats['abnormal_double_slash'] == 0 and feats['subdomain_abuse'] == 0 and feats['has_at_symbol'] == 0 and feats['targets_brand'] == 0:
        is_phishing = False
        avg_prob = min(avg_prob, 0.05)
    
    print(f"URL: {url}")
    print(f"=> Ket luan: {'PHISHING LUA DAO' if is_phishing else 'AN TOAN / LEGITIMATE'}")
    print(f"   - Random Forest Phishing Prob: {prob_rf*100:.2f}%")
    print(f"   - XGBoost Phishing Prob      : {prob_xgb*100:.2f}%")
    print(f"   - SVM Phishing Prob          : {prob_svm*100:.2f}%")
    print(f"   - Muc do nguy hiem trung binh: {avg_prob*100:.2f}%")

predict_url_phishing("http://192.168.1.1/paypal/login.php")
print("-"*50)
predict_url_phishing("https://www.google.com/search?q=cac+thuat+toan+toi+uu")
print("-"*50)
predict_url_phishing("https://google.com")

### 4.2 Kiểm thử Dự đoán Hàng loạt từ File CSV/Excel (Batch Prediction)

In [ ]:
def predict_batch_from_file(input_file_path, output_file_path=None):
    if not os.path.exists(input_file_path):
        print(f"[X] Khong tim thay tep: {input_file_path}")
        return None
    
    ext = os.path.splitext(input_file_path)[1].lower()
    if ext == '.csv':
        df_in = pd.read_csv(input_file_path)
    elif ext in ['.xlsx', '.xls']:
        df_in = pd.read_excel(input_file_path)
    else:
        print("[X] Dinh dang tep khong duoc ho tro!")
        return None
        
    url_col = df_in.columns[0] # Lay cot 1 theo quy dinh
    urls = df_in[url_col].astype(str).tolist()
    
    rf = models["Random Forest"]
    xgb_m = models["XGBoost"]
    svm_m = models["Support Vector Machine (SVM)"]
    
    results = []
    for idx, u in enumerate(urls):
        u_str = u.strip()
        if not u_str:
            continue
        feats = extract_features(u_str)
        df_f = pd.DataFrame([feats])[FEATURE_COLUMNS]
        df_scaled = scaler.transform(df_f)
        
        p_rf = rf.predict_proba(df_f)[0][1]
        p_xgb = xgb_m.predict_proba(df_f)[0][1]
        p_svm = svm_m.predict_proba(df_scaled)[0][1]
        
        avg_p = (p_rf + p_xgb + p_svm) / 3.0
        is_phish = (sum([1 for p in [p_rf, p_xgb, p_svm] if p >= 0.5]) >= 2)
        
        if feats['is_official_domain'] == 1 and feats['has_ip'] == 0 and feats['is_shortened'] == 0 and feats['abnormal_double_slash'] == 0 and feats['subdomain_abuse'] == 0 and feats['has_at_symbol'] == 0 and feats['targets_brand'] == 0:
            is_phish = False
            avg_p = min(avg_p, 0.05)
            
        results.append({
            'STT': idx + 1,
            'URL': u_str,
            'Ket Luan': 'PHISHING (Lua dao)' if is_phish else 'BENIGN (An toan)',
            'Phishing Risk (%)': f"{avg_p*100:.2f}%"
        })
        
    res_df = pd.DataFrame(results)
    print(f"[✓] Da phan loai xong {len(res_df)} URLs!")
    print(f"- Phishing (Lua dao): {(res_df['Ket Luan'] == 'PHISHING (Lua dao)').sum()}")
    print(f"- Benign (An toan): {(res_df['Ket Luan'] == 'BENIGN (An toan)').sum()}")
    
    if output_file_path:
        if output_file_path.endswith('.csv'):
            res_df.to_csv(output_file_path, index=False, encoding='utf-8-sig')
        else:
            res_df.to_excel(output_file_path, index=False)
        print(f"[✓] Da xuat file ket qua ra: {output_file_path}")
        
    return res_df

print('[OK] Da dinh nghia ham predict_batch_from_file thanh cong!')